# trainer-class-skeleton — faded example 2: Trainer validate uses inference_mode context

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `trainer-class-skeleton`. Running the beacon reports progress on the `Trainer: Trainer class skeleton` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: Trainer class skeleton` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`trainer-class-skeleton`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "trainer-class-skeleton"
DD_SUBTOPIC = "Trainer: Trainer class skeleton"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The `validate` method of a Trainer should run under `torch.inference_mode()` (or `torch.no_grad()`), which prevents gradient tracking and reduces memory usage. It also switches the model to eval mode with `model.eval()` so that Dropout and BatchNorm behave correctly for inference.

## Faded exercise 2

Implement `FadedTrainer2`. The `validate` method should switch to eval mode, then iterate the val loader under `t.inference_mode()`, accumulating a batch-size-weighted total loss and a sample count. The blank is the `with t.inference_mode():` context manager that wraps the val loop body.

**Fill in:** The with t.inference_mode(): context block that wraps the validation loop to disable gradient computation.

In [ ]:
import torch as t
import torch.nn as nn

class FadedTrainer2:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        for _ in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total, count = 0.0, 0
        # TODO: The with t.inference_mode(): context block that wraps the validation loop to disable gradient computation.
        # (Replace the next two lines with the proper context-managed block)
        for x, y in self.val_loader:
            loss = None  # TODO: fill this in inside the context
            total = None  # TODO: fill this in inside the context
            count = None  # TODO: fill this in inside the context
        raise NotImplementedError()  # TODO: The with t.inference_mode(): context block that wraps the validation loop to disable gradient computation.


def _test():
    import torch as t
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    t.manual_seed(1)
    X = t.randn(20, 2)
    Y = X[:, 0:1]
    train_dl = DataLoader(TensorDataset(X[:16], Y[:16]), batch_size=8)
    val_dl = DataLoader(TensorDataset(X[16:], Y[16:]), batch_size=4)
    model = nn.Linear(2, 1)
    opt = t.optim.SGD(model.parameters(), lr=0.05)
    trainer = FadedTrainer2(model, opt, train_dl, val_dl, nn.MSELoss())
    trainer.fit(2)
    assert len(trainer.history['val_loss']) == 2
    for v in trainer.history['val_loss']:
        assert v >= 0.0
    # Verify val tensors have no grad_fn (inference_mode was active)
    # We can check indirectly: model should be in eval state after validate
    assert not model.training


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class FadedTrainer2:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        for _ in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total, count = 0.0, 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        self.history['val_loss'].append(total / count)
```
</details>